# Pertemuan 12 - Aktivitas Hands-on: Market Basket Analysis & Rekomendasi Produk  
Nama: Novi Shandi  
NIM: 240401010291  
Kelas: IF401

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    items = list(np.random.choice(produk, n_item, replace=False))
    transaksi.append([str(x) for x in items])

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [['Keju', 'Roti', 'Mentega', 'Kopi', 'Selai'], ['Roti', 'Kopi', 'Teh', 'Selai', 'Mentega'], ['Kopi', 'Susu', 'Teh']]
Jumlah transaksi: 50


Keterangan:  
Kode di atas membuat 50 transaksi belanja sintetis, tiap transaksi berisi 2 sampai 5 produk acak dari 10 produk yang tersedia. Pada 20 transaksi pertama yang mengandung Roti, kode menambahkan Selai jika belum ada, untuk sengaja menciptakan pola pembelian bersama. Hasilnya: 50 transaksi berhasil dibuat, dengan contoh tiga transaksi pertama berisi kombinasi seperti Keju-Roti-Mentega-Kopi-Selai.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print("Shape one-hot:", df.shape)
df.head()

Shape one-hot: (50, 10)


,Gula,Keju,Kopi,Mentega,Roti,Selai,Sereal,Susu,Teh,Telur
0,False,True,True,True,True,True,False,False,False,False
1,False,False,True,True,True,True,False,False,True,False
2,False,False,True,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,True,True
4,True,True,False,True,False,False,False,True,False,False


Keterangan:  
Kode di atas mengubah daftar transaksi menjadi tabel one-hot encoding, tiap kolom mewakili satu produk dengan nilai True/False menandakan ada tidaknya produk itu di transaksi tersebut. Hasilnya: tabel berukuran 50 baris (transaksi) dan 10 kolom (produk), siap diproses algoritma Apriori.

In [ ]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan


Keterangan:  
Kode di atas menguji tiga nilai ambang min_support untuk melihat pengaruhnya terhadap jumlah itemset yang ditemukan. Hasilnya: min_support=0.05 menghasilkan 74 itemset, min_support=0.1 menghasilkan 44 itemset, dan min_support=0.2 menghasilkan 13 itemset. Semakin besar ambang, semakin sedikit itemset yang lolos karena hanya pola yang cukup sering muncul yang dipertahankan. Nilai 0.1 dipilih untuk langkah berikutnya karena menghasilkan jumlah itemset yang wajar (tidak nol, tidak berlebihan).

In [ ]:
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


Keterangan:  
Kode di atas menampilkan 10 itemset paling sering muncul berdasarkan min_support=0.1. Hasilnya: item tunggal terpopuler adalah Selai (support 0,52), Teh (0,46), dan Mentega (0,42). Itemset gabungan pertama yang muncul adalah {Teh, Selai} dengan support 0,24, artinya 24% transaksi mengandung kedua produk itu sekaligus.

In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

         antecedents consequents  support  confidence      lift
10       (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
15  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
9      (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
12     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
13   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


Keterangan:  
Kode di atas membentuk aturan asosiasi dari itemset yang ditemukan, lalu menyaring hanya aturan dengan confidence minimal 0,5 dan lift di atas 1 (menandakan hubungan yang bukan kebetulan). Hasilnya: ditemukan 16 aturan valid. Aturan terkuat adalah {Teh, Keju} → {Telur} dengan lift 2,38 dan confidence 0,857 (85,7% transaksi yang berisi Teh dan Keju juga berisi Telur). Aturan Roti → Selai juga muncul dengan support 0,22, confidence 0,6875, dan lift 1,32, sesuai dengan pola yang sengaja disuntikkan di Langkah 2, dan masuk akal secara bisnis karena roti dan selai memang sering dibeli bersamaan.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
                 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


Keterangan:  
Kode di atas membangun katalog 10 produk beserta kategorinya (Bakery, Dairy, Minuman, Bumbu), lalu menghitung kemiripan antar produk berdasarkan kategori menggunakan cosine similarity. Fungsi rekomendasi_serupa mencari produk dengan kategori paling mirip. Hasilnya: produk yang paling mirip dengan Roti adalah Selai, Sereal, dan Susu, karena Selai dan Sereal berada di kategori Bakery yang sama dengan Roti.

In [ ]:
produk_target = 'Roti'

rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())

print('\nRekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Keterangan:  
Kode di atas membandingkan dua pendekatan rekomendasi untuk produk yang sama, yaitu Roti. Hasilnya: Association Rules merekomendasikan Selai (lift 1,92 dan 1,32, dari dua aturan berbeda), sementara Content-Based Filtering merekomendasikan Selai, Sereal, dan Susu. Kedua pendekatan sama-sama merekomendasikan Selai, menunjukkan hasil yang konsisten meskipun cara kerjanya berbeda: Association Rules menemukan pola dari kebiasaan transaksi, sedangkan Content-Based Filtering menemukan pola dari kemiripan kategori produk.

## Kesimpulan  
1. Algoritma Apriori berhasil menemukan 44 frequent itemset pada min_support=0.1 dan menghasilkan 16 aturan asosiasi yang valid (confidence ≥ 0,5 dan lift > 1).  
2. Aturan asosiasi terkuat adalah {Teh, Keju} → {Telur} dengan lift 2,38, sementara aturan Roti → Selai (yang sengaja disuntikkan ke data) berhasil terdeteksi dengan confidence 0,6875 dan lift 1,32, membuktikan algoritma bekerja dengan benar.  
3. Content-Based Filtering berdasarkan kategori produk berhasil merekomendasikan Selai dan Sereal sebagai produk paling mirip dengan Roti, karena keduanya berada di kategori Bakery yang sama.  
4. Kedua pendekatan rekomendasi (Association Rules dan Content-Based Filtering) menghasilkan rekomendasi yang konsisten untuk Roti, yaitu sama-sama merekomendasikan Selai, meskipun bekerja dengan prinsip yang berbeda: satu berdasarkan pola transaksi bersama, satu lagi berdasarkan kemiripan atribut produk. Dalam praktiknya, kombinasi (hybrid) dari kedua pendekatan ini dapat menghasilkan sistem rekomendasi yang lebih andal.
5. Keterbatasan dan pertanyaan yang muncul: dataset hanya berisi 50 transaksi sintetis yang sebagian besar dibangkitkan secara acak, sehingga aturan dengan Lift tertinggi seperti {Teh, Keju} → {Telur} kemungkinan besar hanya kebetulan statistik dan tidak bermakna secara bisnis. Hanya aturan Roti → Selai yang memang sengaja disuntikkan yang dapat dipercaya. Content-Based Filtering juga baru memakai satu atribut, yaitu kategori produk. Pertanyaan yang muncul: apakah pola yang ditemukan akan bertahan jika diuji pada data transaksi ritel nyata dengan ribuan struk?